## Structure output

##### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

#### Pydantic models provide the richest feature set with field validation, description, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B6924C45F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B69256BC80>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
  title: str = Field(description="the title of the movie")
  year: int = Field(description="This year the movie was released")
  director:str = Field(description="The director of the movie")
  rating:float = Field(description="The movies rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B6924C45F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B69256BC80>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the t

In [4]:
model.invoke("provide details of the movie interstellar")

AIMessage(content='**Interstellar (2014) – Quick Reference Guide**\n\n| Category | Details |\n|----------|---------|\n| **Title** | *Interstellar* |\n| **Release Year** | 2014 |\n| **Genre** | Science‑fiction, Adventure, Drama |\n| **Running Time** | 169 minutes (≈2\u202fh\u202f49\u202fmin) |\n| **MPAA Rating** | PG‑13 (Violence, some intense scenes, language) |\n| **Director** | Christopher Nolan |\n| **Screenwriter(s)** | Jonathan Nolan (story), Christopher Nolan (screenplay) |\n| **Producers** | Christopher Nolan, Lynda Obst, Emma Thomas, Kipp Tribble |\n| **Cinematography** | Hoyte van Hoytema |\n| **Music** | Hans Zimmer (original score) |\n| **Production Companies** | Syncopy Inc., Lynda Obst Productions, Paramount Pictures, Warner Bros. Pictures |\n| **Distributor** | Paramount Pictures (U.S.), Warner Bros. (International) |\n| **Budget** | Approx. $165\u202fmillion |\n| **Box‑Office Gross** | $677\u202fmillion worldwide |\n| **Release Dates** | • Premiere: October\u202f30\u202f

In [5]:
response = model_with_structure.invoke("provide details of the anime movie weathering with you")
response

Movie(title='Weathering with You', year=2019, director='Makoto Shinkai', rating=8.5)

### Nested Structure

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
  name: str
  role: str

class MovieDetails(BaseModel):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details of the movie interstellar")
response

MovieDetails(title='Interstellar', year=2014, cast=[Actor(name='Matthew McConaughey', role='Cooper'), Actor(name='Anne Hathaway', role='Brand'), Actor(name='Jessica Chastain', role='Murph'), Actor(name='Michael Caine', role='Professor Brand'), Actor(name='Matt Damon', role='Dr. Mann')], genres=['Adventure', 'Drama', 'Sci-Fi'], budget=165000000.0)

## TypedDict

#### TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [7]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
  """A movie with details"""
  title: Annotated[str, ..., "The title of the movie"]
  year: Annotated[int, ..., "The year the movie was released"]
  director: Annotated[str, ..., "The director of the movie"]
  rating: Annotated[float, ..., "The rating of the movie out of 10"]

model_with_typedDict = model.with_structured_output(MovieDict)
response = model_with_typedDict.invoke("provide details of the movie Avengers: Endgame")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [8]:
class Actor(TypedDict):
  name: str
  role: str

class MovieDetails(TypedDict):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details of the movie interstellar")
response

{'budget': 165000000,
 'cast': [{'name': 'Matthew McConaughey', 'role': 'Cooper'},
  {'name': 'Anne Hathaway', 'role': 'Brand'},
  {'name': 'Jessica Chastain', 'role': 'Murph'},
  {'name': 'Michael Caine', 'role': 'Professor Brand'},
  {'name': 'Mackenzie Foy', 'role': 'Young Murph'},
  {'name': 'Casey Affleck', 'role': 'Tom'},
  {'name': 'Matt Damon', 'role': 'Dr. Mann'},
  {'name': 'David Gyasi', 'role': 'Romilly'},
  {'name': 'Bill Irwin', 'role': 'TARS (voice)'},
  {'name': 'Josh Stewart', 'role': 'Doyle'}],
 'genres': ['Adventure', 'Drama', 'Sci-Fi'],
 'title': 'Interstellar',
 'year': 2014}

In [9]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
##### A data class is a class typically containing mainly data, although there aren't really any restrictions. You create iti using the @dataclass decorator

In [10]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
# Pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
  """Contact information for a person"""
  name: str = Field(description="The name of the person")
  email: str = Field(description="The email address of the person")
  phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model= "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from: John Doe, john@example.com, 123-456-7890"
    }]
})
print(result,"\n")

print(result["structured_response"]) 

{'messages': [HumanMessage(content='Extract contact information from: John Doe, john@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='6ddf887d-d0bc-4634-9e21-7189ebdb16c6'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"123-456-7890"}', additional_kwargs={'reasoning_content': 'The user wants extraction of contact info from given text. We need to output JSON matching the ContactInfo schema: fields name, email, phone, all required. Provide compact JSON.\n\nSo output: {"name":"John Doe","email":"john@example.com","phone":"123-456-7890"}\n\nMake sure it\'s compact, no extra whitespace.'}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 237, 'total_tokens': 341, 'completion_time': 0.218831452, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.008954353, 'prompt_tokens_details': None, 'queue_time': 0.380204289, 'total_time': 0.227785805}, 'model_name': 'openai/gpt-oss-120b', 'system_fin

In [ ]:
# TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
  """Contact information for a person"""
  name: str  # The name of the person
  email: str  # The email address of the person
  phone: str  # The phone number of the person

agent = create_agent(
    model= "groq:openai/gpt-oss-120b",
    response_format=ContactInfo 
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from: John Doe, john@example.com, 123-456-7890"
    }]
})
print(result,"\n")

print(result["structured_response"]) 

{'messages': [HumanMessage(content='Extract contact information from: John Doe, john@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='985155fe-d2e4-4bf2-8c54-9d13c23ca8bb'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"123-456-7890"}', additional_kwargs={'reasoning_content': 'The user asks: "Extract contact information from: John Doe, john@example.com, 123-456-7890". We need to output JSON matching the ContactInfo schema. Must include fields name, email, phone. Provide compact JSON. So output: {"name":"John Doe","email":"john@example.com","phone":"123-456-7890"} Ensure valid JSON, no extra fields.'}, response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 211, 'total_tokens': 327, 'completion_time': 0.242269598, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.009789132, 'prompt_tokens_details': None, 'queue_time': 0.309183197, 'total_time': 0.25205873}, 'model_name': 'openai/gpt-oss-12

In [13]:
# DataClass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
  """Contact information for a person"""
  name: str  # The name of the person
  email: str  # The email address of the person
  phone: str  # The phone number of the person

agent = create_agent(
    model= "groq:openai/gpt-oss-120b",
    response_format=ContactInfo 
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from: John Doe, john@example.com, 123-456-7890"
    }]
})
print(result,"\n")

print(result["structured_response"]) 

{'messages': [HumanMessage(content='Extract contact information from: John Doe, john@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='40c3d45a-e5b3-4a87-a81a-0dd9e09b069a'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"123-456-7890"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact information. The response format is JSON schema ContactInfo. Must output JSON with fields name, email, phone. Provide compact JSON. So: {"name":"John Doe","email":"john@example.com","phone":"123-456-7890"} Ensure valid JSON.'}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 211, 'total_tokens': 305, 'completion_time': 0.199951719, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.008543311, 'prompt_tokens_details': None, 'queue_time': 0.379511888, 'total_time': 0.20849503}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand